# Credit-risk feature analysis

This notebook asks which application-time features contain useful signal for `default_12m` before any multivariate model is trained.

It intentionally uses **only the training split** produced by `01_data_preparation.ipynb`. Validation and test data remain untouched for later model selection and final evaluation.

The analysis covers:

- target balance;
- missingness associated with default;
- numeric point-biserial correlation, univariate ROC-AUC and mutual information;
- categorical Cramer's V, risk-rate spread and mutual information; and
- highly correlated numeric pairs that may be redundant.

These are association and standalone-signal diagnostics, not causal conclusions or final feature-selection decisions. Gender and age remain outside model features and are reserved for fairness analysis.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import OrdinalEncoder

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

cwd = Path.cwd().resolve()
NOTEBOOK_DIR = cwd if cwd.name == "notebooks" else cwd / "notebooks"
DATA_PREP_NOTEBOOK = NOTEBOOK_DIR / "01_data_preparation.ipynb"

if not DATA_PREP_NOTEBOOK.exists():
    raise FileNotFoundError(f"Data-preparation notebook not found: {DATA_PREP_NOTEBOOK}")

# Reproduce target construction, chronological splitting and preparation exactly once.
get_ipython().run_line_magic("run", f'"{DATA_PREP_NOTEBOOK}"')


c:\Users\User\Documents\UNI\credit_risk_system\.venv\Lib\site-packages\nbformat\__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


Project root: C:\Users\User\Documents\UNI\credit_risk_system
Dataset path: C:\Users\User\Documents\UNI\credit_risk_system\data\LoanData_Bondora.csv


FileNotFoundError: Bondora CSV not found at C:\Users\User\Documents\UNI\credit_risk_system\data\LoanData_Bondora.csv. Place it in data/ or set BONDORA_DATA_PATH.

FileNotFoundError: Bondora CSV not found at C:\Users\User\Documents\UNI\credit_risk_system\data\LoanData_Bondora.csv. Place it in data/ or set BONDORA_DATA_PATH.

In [ ]:
required_objects = [
    "X_train",
    "X_validation",
    "X_test",
    "y_train",
    "y_validation",
    "y_test",
]
missing_objects = [name for name in required_objects if name not in globals()]
if missing_objects:
    raise RuntimeError(f"Data preparation did not create: {missing_objects}")

assert X_train.index.equals(y_train.index)
assert TARGET not in X_train.columns
assert set(PROTECTED_COLUMNS).isdisjoint(X_train.columns)

print(f"Training rows available for feature analysis: {len(X_train):,}")
print(f"Features before one-hot encoding: {X_train.shape[1]}")


## 1. Target balance

Class balance determines how strongly accuracy can mislead us later. This notebook reports the training prevalence but does not resample or choose class weights; those decisions belong inside model validation.


In [ ]:
target_balance = pd.DataFrame(
    {
        "count": y_train.value_counts().sort_index(),
        "percentage": y_train.value_counts(normalize=True).sort_index().mul(100),
    }
).rename(index={0: "no default within 12m", 1: "default within 12m"})
target_balance


In [ ]:
numeric_columns = X_train.select_dtypes(include="number").columns.tolist()
categorical_columns = X_train.select_dtypes(exclude="number").columns.tolist()

pd.Series(
    {
        "numeric_features": len(numeric_columns),
        "categorical_features": len(categorical_columns),
        "total_features": X_train.shape[1],
    },
    name="count",
)


## 2. Missingness as potential signal

Missing values can be informative—for example, an unreported field may reflect a particular application process. For every feature with missing values, compare the default rate when the value is present versus missing.

A large difference is a reason to preserve a missingness indicator, but it may also reveal a data-collection or policy artifact that requires investigation.


In [ ]:
missingness_rows = []

for feature in X_train.columns:
    missing = X_train[feature].isna()
    if not missing.any() or missing.all():
        continue

    missingness_rows.append(
        {
            "feature": feature,
            "missing_count": int(missing.sum()),
            "missing_pct": missing.mean() * 100,
            "default_rate_missing": y_train.loc[missing].mean(),
            "default_rate_present": y_train.loc[~missing].mean(),
            "absolute_rate_gap": abs(
                y_train.loc[missing].mean() - y_train.loc[~missing].mean()
            ),
        }
    )

missingness_signal = (
    pd.DataFrame(missingness_rows)
    .set_index("feature")
    .sort_values("absolute_rate_gap", ascending=False)
)
missingness_signal.head(20)


## 3. Numeric feature signal

Three complementary measures are used:

- **Absolute point-biserial correlation:** linear association with the binary target.
- **Direction-free univariate ROC-AUC:** how well one feature ranks defaulters above non-defaulters; `0.5` means no ranking signal and `1.0` means perfect separation. The direction column tells us whether higher or lower values correspond to default.
- **Mutual information:** captures broader, potentially nonlinear dependency; zero means no detected dependency.

All medians and measurements below are calculated using training data only.


In [ ]:
numeric_raw = X_train[numeric_columns].apply(pd.to_numeric, errors="coerce")
usable_numeric = [
    feature
    for feature in numeric_columns
    if numeric_raw[feature].notna().any() and numeric_raw[feature].nunique(dropna=True) > 1
]

numeric_medians = numeric_raw[usable_numeric].median()
numeric_imputed = numeric_raw[usable_numeric].fillna(numeric_medians)

numeric_mi = {}
for feature in usable_numeric:
    is_discrete = numeric_imputed[feature].nunique() <= 10
    if is_discrete:
        feature_values = pd.factorize(numeric_imputed[feature], sort=True)[0].reshape(-1, 1)
    else:
        feature_values = numeric_imputed[[feature]]
    numeric_mi[feature] = mutual_info_classif(
        feature_values,
        y_train,
        discrete_features=[is_discrete],
        random_state=42,
    )[0]

numeric_rows = []
for feature in usable_numeric:
    values = numeric_imputed[feature]
    raw_auc = roc_auc_score(y_train, values)
    correlation = values.corr(y_train)
    numeric_rows.append(
        {
            "feature": feature,
            "point_biserial_correlation": correlation,
            "absolute_correlation": abs(correlation),
            "univariate_auc": raw_auc,
            "direction_free_auc": max(raw_auc, 1 - raw_auc),
            "direction": "higher -> default" if raw_auc >= 0.5 else "lower -> default",
            "mutual_information": numeric_mi[feature],
        }
    )

numeric_signal = (
    pd.DataFrame(numeric_rows)
    .set_index("feature")
    .sort_values(["direction_free_auc", "mutual_information"], ascending=False)
)
numeric_signal.head(20)


In [ ]:
# Compare the training distributions behind the signal scores.
numeric_target_profiles = (
    X_train[usable_numeric]
    .assign(**{TARGET: y_train})
    .groupby(TARGET)
    .agg(["median", "mean", "std"])
)
numeric_target_profiles


## 4. Numeric redundancy

Strongly correlated predictors may carry nearly the same information. This does not automatically mean one must be removed—regularized linear models and tree models handle redundancy differently—but the pairs should be reviewed before modelling and explanation.


In [ ]:
spearman = numeric_imputed.corr(method="spearman")
upper_triangle = np.triu(np.ones(spearman.shape, dtype=bool), k=1)

redundant_pairs = (
    spearman.where(upper_triangle)
    .stack()
    .rename("spearman_correlation")
    .reset_index()
    .rename(columns={"level_0": "feature_a", "level_1": "feature_b"})
)
redundant_pairs["absolute_correlation"] = redundant_pairs["spearman_correlation"].abs()
redundant_pairs = redundant_pairs.loc[
    redundant_pairs["absolute_correlation"] >= 0.80
].sort_values("absolute_correlation", ascending=False)

redundant_pairs


## 5. Categorical feature signal

Categorical association is measured with:

- **Bias-corrected Cramer's V:** overall association with the binary target (`0` to `1`).
- **Default-rate spread:** difference between the highest- and lowest-risk sufficiently populated categories.
- **Mutual information:** dependency between the category and target.

Rare categories below the minimum count are excluded only from the risk-spread calculation so that a category containing one or two defaults cannot dominate the result.


In [ ]:
MIN_CATEGORY_COUNT = max(50, int(0.005 * len(X_train)))
categorical_filled = (
    X_train[categorical_columns]
    .astype("object")
    .where(X_train[categorical_columns].notna(), "__MISSING__")
    .astype(str)
)


def corrected_cramers_v(feature: str) -> float:
    contingency = pd.crosstab(categorical_filled[feature], y_train)
    if min(contingency.shape) < 2:
        return 0.0

    chi2 = chi2_contingency(contingency, correction=False)[0]
    n = contingency.to_numpy().sum()
    rows, columns = contingency.shape
    phi2 = chi2 / n
    phi2_corrected = max(0.0, phi2 - ((columns - 1) * (rows - 1)) / (n - 1))
    rows_corrected = rows - ((rows - 1) ** 2) / (n - 1)
    columns_corrected = columns - ((columns - 1) ** 2) / (n - 1)
    denominator = min(columns_corrected - 1, rows_corrected - 1)
    return float(np.sqrt(phi2_corrected / denominator)) if denominator > 0 else 0.0


category_encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
categorical_encoded = category_encoder.fit_transform(categorical_filled)
categorical_mi = mutual_info_classif(
    categorical_encoded,
    y_train,
    discrete_features=True,
    random_state=42,
)

categorical_rows = []
for position, feature in enumerate(categorical_columns):
    grouped = pd.DataFrame({"category": categorical_filled[feature], TARGET: y_train}).groupby(
        "category"
    )[TARGET].agg(["count", "mean"])
    supported = grouped.loc[grouped["count"] >= MIN_CATEGORY_COUNT]
    rate_spread = supported["mean"].max() - supported["mean"].min() if len(supported) >= 2 else np.nan

    categorical_rows.append(
        {
            "feature": feature,
            "unique_categories": categorical_filled[feature].nunique(),
            "cramers_v": corrected_cramers_v(feature),
            "supported_default_rate_spread": rate_spread,
            "mutual_information": categorical_mi[position],
        }
    )

categorical_signal = (
    pd.DataFrame(categorical_rows)
    .set_index("feature")
    .sort_values(["cramers_v", "mutual_information"], ascending=False)
)
categorical_signal


In [ ]:
def category_default_rates(feature: str, min_count: int = MIN_CATEGORY_COUNT) -> pd.DataFrame:
    if feature not in categorical_columns:
        raise ValueError(f"{feature!r} is not a categorical training feature.")

    rates = pd.DataFrame(
        {"category": categorical_filled[feature], TARGET: y_train}
    ).groupby("category")[TARGET].agg(["count", "mean"])
    return (
        rates.rename(columns={"mean": "default_rate"})
        .loc[lambda frame: frame["count"] >= min_count]
        .sort_values("default_rate", ascending=False)
    )


top_categorical_feature = categorical_signal.index[0]
print(f"Highest categorical association: {top_categorical_feature}")
category_default_rates(top_categorical_feature).head(20)


## 6. Train-only feature shortlist

The tables below combine rankings within each feature type. They are a shortlist for the modelling notebook—not an instruction to delete every lower-ranked feature.

A feature should be removed only after considering:

1. whether it is genuinely available at prediction time;
2. whether its signal is stable on validation data;
3. whether it duplicates another feature;
4. whether it creates fairness, policy or explainability concerns; and
5. whether an ablation test shows that the model does not need it.


In [ ]:
numeric_shortlist = numeric_signal.copy()
numeric_shortlist["average_signal_rank"] = numeric_shortlist[
    ["absolute_correlation", "direction_free_auc", "mutual_information"]
].rank(ascending=False).mean(axis=1)
numeric_shortlist = numeric_shortlist.sort_values("average_signal_rank")

categorical_shortlist = categorical_signal.copy()
categorical_shortlist["average_signal_rank"] = categorical_shortlist[
    ["cramers_v", "supported_default_rate_spread", "mutual_information"]
].rank(ascending=False).mean(axis=1)
categorical_shortlist = categorical_shortlist.sort_values("average_signal_rank")

print("Top numeric candidates")
display(numeric_shortlist.head(15))

print("Top categorical candidates")
display(categorical_shortlist.head(15))


## Handoff to modelling

The modelling notebook should now:

1. establish a Logistic Regression baseline;
2. keep preprocessing inside each fitted pipeline;
3. compare candidate models using validation data;
4. evaluate probability calibration;
5. use ablation or permutation importance to confirm that shortlisted features add value jointly; and
6. reserve the test split for one final unbiased evaluation.

Do not select the final model, decision threshold or risk bands from the training associations alone.
